In [91]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
import torchbnn as bnn
from tqdm.auto import tqdm


# Установка случайного сида для воспроизводимости
torch.manual_seed(42)
np.random.seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"

In [92]:
df = pd.read_csv('../data/data.csv')
df

,"Sigma, Mpa",T.K,t.h,Th,C,Cr,Co,Mo,W,Al,...,La,S,Si,Mn,P,Hf,Cu,Ge,Ga,Ni
0,241.316495,1144.2600,4.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.50,NaN,NaN,NaN,99.500
1,241.316495,1144.2600,113.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.75,NaN,NaN,NaN,99.250
2,241.316495,1144.2600,68.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.50,NaN,NaN,0.05,99.450
3,241.316495,1144.2600,40.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.50,NaN,NaN,0.20,99.300
4,241.316495,1144.2600,32.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.50,NaN,NaN,0.50,99.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3441,1061.792578,922.0389,1183.7,NaN,0.02,17.0,28.4,3.4,1.9,1.03,...,NaN,6.0,0.05,0.05,NaN,NaN,0.02,NaN,NaN,33.904
3442,1103.161120,866.4833,25554.7,NaN,0.02,17.0,28.4,3.4,1.9,1.03,...,NaN,6.0,0.05,0.05,NaN,NaN,0.02,NaN,NaN,33.904
3443,241.316495,1199.8170,183.2,NaN,0.18,10.0,15.0,3.0,NaN,5.50,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,59.750
3444,241.316495,1199.8170,153.0,NaN,0.18,10.0,15.0,3.0,NaN,5.50,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,59.750


In [93]:
mass = '232,03806	12,011	51,9961	58,933194	95,95	183,84	26,9815385	47,867	92,90637	10,81	55,845	88,90584	91,224	180,94788	186,207	101,07	50,9415	140,116	138,90547	32,06	28,085	54,938044	24,305	30,973761998	178,49	107,8682	63,546	208,9804	207,2	192,22	72,63	69,723	58,6934'.replace(',', '.').split()
element = 'Th	C	Cr	Co	Mo	W	Al	Ti	Nb	B	Fe	Y	Zr	Ta	Re	Ru	V	Ce	La	S	Si	Mn	Mg	P	Hf	Ag	Cu	Bi	Pb	Ir	Ge	Ga	Ni'.split()
atom_mass = dict(zip(element, mass))

In [94]:
atom_mass['Ni']

'58.6934'

In [95]:
for elem in df.drop(columns=['Sigma, Mpa', 'T.K', 't.h']).columns.to_list():
    df[elem] = df[elem] / float(atom_mass[elem])
df['sum'] = df[df.drop(columns=['Sigma, Mpa', 'T.K', 't.h']).columns.to_list()].sum(axis=1, skipna=True)
df['PLM'] = df['T.K'] * (20 + np.log10(df['t.h'])) * 1e-5

cols = df.drop(columns=['Sigma, Mpa', 'T.K', 't.h']).columns
df.loc[:, cols] = df.loc[:, cols].div(df['sum'], axis=0)
df.loc[:, cols] = df.loc[:, cols].div(df['Ni'], axis=0)



df = df.fillna(0)

df = df.drop(columns=['T.K', 't.h', 'Ni', 'sum'])

df

,"Sigma, Mpa",Th,C,Cr,Co,Mo,W,Al,Ti,Nb,...,La,S,Si,Mn,P,Hf,Cu,Ge,Ga,PLM
0,241.316495,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.001652,0.000000,0.0,0.000000,0.139405
1,241.316495,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.002485,0.000000,0.0,0.000000,0.149239
2,241.316495,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.001653,0.000000,0.0,0.000423,0.147465
3,241.316495,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.001656,0.000000,0.0,0.001695,0.146154
4,241.316495,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.001661,0.000000,0.0,0.004252,0.145925
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3441,1061.792578,0.0,0.002883,0.566000,0.834251,0.061344,0.017892,0.066086,0.112115,0.020497,...,0.0,0.323986,0.003082,0.001576,0.0,0.000000,0.000545,0.0,0.000000,0.368295
3442,1103.161120,0.0,0.002883,0.566000,0.834251,0.061344,0.017892,0.066086,0.112115,0.020497,...,0.0,0.323986,0.003082,0.001576,0.0,0.000000,0.000545,0.0,0.000000,0.366118
3443,241.316495,0.0,0.014721,0.188921,0.250025,0.030713,0.000000,0.200238,0.112870,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.262391
3444,241.316495,0.0,0.014721,0.188921,0.250025,0.030713,0.000000,0.200238,0.112870,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.261469


In [96]:
def y_from_sigma(sigma_mpa):
    # y = -log10(sigma)
    return -np.log10(sigma_mpa)

def sigma_from_y(y):
    # sigma = 10^(-y)
    return 10 ** (-y)

def calculate_mse(y_pred, y_true):
    """Вычисление MSE (Mean Squared Error)"""
    return torch.mean((y_pred - y_true) ** 2).item()

def calculate_rmse_absolute(sigma_pred, sigma_real):
    """
    Вычисление RMSE для физических значений σ (MPa)
    Корень из среднего квадрата абсолютных ошибок
    """
    mse = torch.mean((sigma_pred - sigma_real) ** 2).item()
    return np.sqrt(mse)

def calculate_rmse_relative(sigma_pred, sigma_real):
    """
    Вычисление RMSE согласно формуле (6) из статьи.
    Это относительная ошибка (Root Mean Square Relative Error)
    """
    epsilon = 1e-8
    relative_error = (sigma_pred - sigma_real) / (sigma_real.abs() + epsilon)
    mse_relative = torch.mean(relative_error ** 2).item()
    return np.sqrt(mse_relative)

In [97]:
target_col = 'Sigma, Mpa'

y = np.asarray(y_from_sigma(df[target_col]), dtype=np.float32).reshape(-1, 1).astype("float32")
X = df.copy().drop(columns=[target_col]).to_numpy().astype("float32")
X, y

(array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
         0.13940506],
        [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
         0.1492392 ],
        [0.        , 0.        , 0.        , ..., 0.        , 0.00042323,
         0.14746492],
        ...,
        [0.        , 0.01472125, 0.18892115, ..., 0.        , 0.        ,
         0.2623908 ],
        [0.        , 0.01472125, 0.18892115, ..., 0.        , 0.        ,
         0.26146874],
        [0.        , 0.01472125, 0.18892115, ..., 0.        , 0.        ,
         0.2588929 ]], shape=(3446, 27), dtype=float32),
 array([[-2.382587],
        [-2.382587],
        [-2.382587],
        ...,
        [-2.382587],
        [-2.382587],
        [-2.440579]], shape=(3446, 1), dtype=float32))

In [98]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.85,
    random_state=42,
    shuffle=True
)


In [99]:
class BRANN(nn.Module):
    """
    Bayesian Regularized Artificial Neural Network
    Реализует настоящую байесовскую регуляризацию с α и β
    """
    def __init__(self, input_size=24, hidden_size=13):
        super(BRANN, self).__init__()
        self.hidden = nn.Linear(input_size, hidden_size)
        self.activation = nn.Tanh()
        self.output = nn.Linear(hidden_size, 1)
        
        # Инициализация весов малыми значениями
        self._initialize_weights()
    
    def _initialize_weights(self):
        """Инициализация весов"""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, mean=0, std=0.1)
                nn.init.zeros_(m.bias)
    
    def forward(self, x):
        x = self.hidden(x)
        x = self.activation(x)
        x = self.output(x)
        return x
    
    def get_all_weights(self):
        """Получить все веса сети как один вектор"""
        weights = []
        for param in self.parameters():
            weights.append(param.view(-1))
        return torch.cat(weights)
    
    def compute_ed(self, X, y):
        """
        E_D - ошибка данных (сумма квадратов ошибок)
        E_D = Σ[y_i - f(X_i)]²
        """
        predictions = self.forward(X)
        ed = torch.sum((predictions - y) ** 2)
        return ed
    
    def compute_ew(self):
        """
        E_W - ошибка весов (сумма квадратов весов)
        E_W = Σw_j²
        """
        ew = 0.0
        for param in self.parameters():
            ew += torch.sum(param ** 2)
        return ew
    
    def compute_regularized_loss(self, ed, ew, alpha, beta):
        """
        S(w) = β·E_D + α·E_W
        Формула (17) из статьи
        """
        return beta * ed + alpha * ew

In [100]:
def compute_effective_parameters(model, alpha, beta, device):
    """
    Вычисление эффективного количества параметров γ
    Формула (28): γ = N_W - α·trace(G^(-1))
    
    G - гессиан функции S(w)
    G ≈ β·H + α·I, где H - гессиан E_D
    """
    # Получаем все веса
    w = model.get_all_weights()
    n_weights = len(w)
    
    # Приближенное вычисление trace(G^(-1)) через собственные значения
    # Для простоты используем диагональное приближение гессиана
    
    # Вычисляем диагональ гессиана (приближенно)
    hessian_diag = []
    for param in model.parameters():
        # Для квадратичной функции: H_ii ≈ 2β + 2α
        # Но более точно можно вычислить через вторые производные
        grad = torch.ones_like(param)
        hessian_diag.append(torch.ones_like(param) * (2 * beta + 2 * alpha))
    
    # Объединяем диагональ
    hessian_diag_flat = torch.cat([h.view(-1) for h in hessian_diag])
    
    # trace(G^(-1)) ≈ Σ(1/G_ii)
    trace_inv = torch.sum(1.0 / (hessian_diag_flat + 1e-8))
    
    # γ = N_W - α·trace(G^(-1))
    gamma = n_weights - alpha * trace_inv.item()
    gamma = max(0, min(gamma, n_weights))  # Ограничиваем γ
    
    return gamma, n_weights

## train brann

In [101]:
def train_brann(X_train, y_train, input_size=24, max_epochs=100, alpha_init=0.01, beta_init=1.0):
    """
    Обучение BRANN с автоматической настройкой α и β
    Согласно формулам (28) из статьи
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = BRANN(input_size).to(device)
    
    X_train_t = torch.FloatTensor(X_train).to(device)
    y_train_t = torch.FloatTensor(y_train).view(-1, 1).to(device)
    
    # Инициализация гиперпараметров
    alpha = alpha_init  # регуляризация весов
    beta = beta_init    # точность данных
    
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    
    print(f"  Начальные значения: α={alpha:.6f}, β={beta:.6f}")
    
    history = {
        'alpha': [],
        'beta': [],
        'gamma': [],
        'ed': [],
        'ew': [],
        'loss': []
    }
    
    for epoch in range(max_epochs):
        model.train()
        optimizer.zero_grad()
        
        # Вычисляем E_D и E_W
        ed = model.compute_ed(X_train_t, y_train_t)
        ew = model.compute_ew()
        
        # Регуляризованная функция ошибок
        loss = model.compute_regularized_loss(ed, ew, alpha, beta)
        
        # Обратное распространение
        loss.backward()
        optimizer.step()
        
        # Обновляем α и β каждые 10 эпох (или чаще для лучшей сходимости)
        if (epoch + 1) % 5 == 0 or epoch == 0:
            # Вычисляем эффективное количество параметров γ
            gamma, n_weights = compute_effective_parameters(model, alpha, beta, device)
            
            # Обновляем α и β согласно формулам (28)
            # α_new = γ / (2·E_W)
            # β_new = (N_D - γ) / (2·E_D)
            n_data = len(X_train)
            
            ew_val = ew.item()
            ed_val = ed.item()
            
            # Избегаем деления на ноль
            if ew_val > 1e-10:
                alpha_new = gamma / (2 * ew_val)
            else:
                alpha_new = alpha
            
            if ed_val > 1e-10:
                beta_new = max(0, (n_data - gamma) / (2 * ed_val))
            else:
                beta_new = beta
            
            # Ограничиваем α и β разумными пределами
            alpha_new = np.clip(alpha_new, 1e-10, 1e10)
            beta_new = np.clip(beta_new, 1e-10, 1e10)
            
            alpha = alpha_new
            beta = beta_new
            
            # Сохраняем историю
            history['alpha'].append(alpha)
            history['beta'].append(beta)
            history['gamma'].append(gamma)
            history['ed'].append(ed_val)
            history['ew'].append(ew_val)
            history['loss'].append(loss.item())
            
            # Выводим прогресс
    print(pd.DataFrame(history, index=range(0, 105, 5)))
    
    print(f"\n  Финальные значения: α={alpha:.6f}, β={beta:.6f}, γ={gamma:.1f}")
    print(f"  Отношение β/α = {beta/alpha:.2f} (баланс данных и регуляризации)")
    
    return model, history

In [102]:
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_scaled = scaler_X.fit_transform(X_temp).astype("float32")
y_scaled = scaler_y.fit_transform(y_temp).astype("float32").reshape(-1,1)

X_test_scaled = scaler_X.transform(X_test).astype("float32")
y_test_scaled = scaler_y.transform(y_test).astype("float32").reshape(-1,1)

# 2. Ансамбль из 7 сетей (BRANN / Bootstrap procedure)
# Текст: "The modeling ensemble consists of seven networks that act in parallel."
n_ensemble = 7
models = []



for i in range(n_ensemble):
    print(f"\nОбучение сети #{i+1}:")
    
    model, history = train_brann(
        X_scaled, y_scaled,
        input_size=27,
        max_epochs=100,
        alpha_init=0.01,
        beta_init=1.0
    )



Обучение сети #1:
  Начальные значения: α=0.010000, β=1.000000
             alpha      beta       gamma          ed        ew        loss
0        52.057266  0.133263  376.128713  524.795532  3.612644  524.831665
5        36.348377  0.355026  189.482577  459.849487  2.606479  196.966934
10       68.086342  0.375822  190.828159  432.614502  1.401369  204.527054
15      113.003179  0.402630  190.037526  404.791199  0.840850  209.379745
20      171.648255  0.409977  189.671018  397.984558  0.552499  222.674835
25      266.069109  0.383121  189.450344  426.170624  0.356017  235.829849
30      515.197116  0.348199  189.271762  469.168945  0.183689  228.622269
35     1534.407368  0.322209  189.127644  507.236938  0.061629  208.370377
40     7101.155956  0.316564  189.039667  516.421082  0.013310  186.819092
45    21553.930819  0.316728  189.008466  516.203003  0.004385  194.546494
50    43840.474067  0.316867  189.002775  515.984924  0.002156  209.887787
55    70977.025975  0.316871  189.00

In [103]:

X_test_t = torch.FloatTensor(X_test_scaled)
y_test_t = torch.FloatTensor(y_test_scaled).view(-1, 1)

predictions_list = []

with torch.no_grad():
    y_pred_scaled = model(X_test_t)


# Обратная нормализация y
y_pred = scaler_y.inverse_transform(y_pred_scaled)

# Обратная логарифмическая трансформация (формула 4)
sigma_pred = sigma_from_y(y_pred)
sigma_pred_tensor = torch.FloatTensor(sigma_pred).view(-1, 1)
sigma_real_tensor = torch.FloatTensor(y_test_t).view(-1, 1)

# Расчет ошибок
rmse_absolute = calculate_rmse_absolute(sigma_pred_tensor, sigma_real_tensor)
rmse_relative = calculate_rmse_relative(sigma_pred_tensor, sigma_real_tensor)

# MAPE (Mean Absolute Percentage Error)
mape = torch.mean(torch.abs((sigma_pred_tensor - sigma_real_tensor) / 
                                (sigma_real_tensor + 1e-8))) * 100

print(f"\nРезультаты:")
print(f"  RMSE (абсолютная): {rmse_absolute:.2f} MPa")
print(f"  RMSE (относительная, формула 6): {rmse_relative:.4f}")
print(f"  MAPE: {mape.item():.2f}%")
print(f"  Мин. предсказанное σ: {sigma_pred.min():.2f} MPa")
print(f"  Макс. предсказанное σ: {sigma_pred.max():.2f} MPa")


Результаты:
  RMSE (абсолютная): 253.42 MPa
  RMSE (относительная, формула 6): 9097.3574
  MAPE: 135720.48%
  Мин. предсказанное σ: 253.32 MPa
  Макс. предсказанное σ: 253.70 MPa
